In [3]:
cd /data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY/

/data/home/wangzz_group/zhaipengyuan/BEPH-main/DATA_DIRECTORY


/data/home/wangzz_group/zhaipengyuan/.conda/envs/BEPH/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 处理全部数据

In [4]:
import os
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from PIL import Image
from tqdm import tqdm

# ================= 配置区域 =================
ROOT_DIR = './kz_data' 
H5AD_DIR = os.path.join(ROOT_DIR, 'Raw_Data', 'Log1p')
IMAGE_DIR = os.path.join(ROOT_DIR, 'Raw_Data', 'images')
CSV_PATH = 'process_list.csv'

OUTPUT_DIR = os.path.join(ROOT_DIR, 'Segmentation')
PATCHES_SAVE_DIR = os.path.join(OUTPUT_DIR, 'patches')

SRC_SIZE = 80
TARGET_SIZE = 224

# ================= 修复后的处理函数 =================
def process_single_sample(slide_id, h5ad_path, image_path):
    if not os.path.exists(h5ad_path) or not os.path.exists(image_path):
        return False

    try:
        adata = sc.read_h5ad(h5ad_path)
        Image.MAX_IMAGE_PIXELS = None 
        full_image = Image.open(image_path)
        img_w, img_h = full_image.size

        if 'spatial' not in adata.obsm.keys():
            return False

        coords = adata.obsm['spatial']
        
        # --- 🛠️ 关键修改开始：更强健的缩放因子获取逻辑 ---
        scale_factor = 1.0
        library_id = list(adata.uns.get('spatial', {}).keys())[0] if adata.uns.get('spatial') else None
        
        if library_id:
            scalefactors = adata.uns['spatial'][library_id].get('scalefactors', {})
            
            # 【强制逻辑】我们默认使用的图片就是 hires 图片
            # 不再通过文件名判断，而是直接取 tissue_hires_scalef
            if 'tissue_hires_scalef' in scalefactors:
                scale_factor = scalefactors['tissue_hires_scalef']
                print(f"   Using hires scale factor: {scale_factor:.4f}")
            elif 'tissue_lowres_scalef' in scalefactors:
                # 如果没有hires因子，尝试用lowres（极少见）
                scale_factor = scalefactors['tissue_lowres_scalef']
                print(f"   Warning: Using lowres scale factor: {scale_factor:.4f}")
        else:
            print("   ⚠️ Warning: No spatial metadata found in uns. Using scale=1.0")
        # ------------------------------------------------

        # 计算像素坐标
        pixel_coords = (coords * scale_factor).astype(int)
        top_left_coords = pixel_coords - (SRC_SIZE // 2)
        
        # --- 🛠️ 调试打印 (只打印第一个点) ---
        if len(coords) > 0:
            print(f"   [Debug] 图片尺寸: {img_w}x{img_h}")
            print(f"   [Debug] 原始坐标示例: {coords[0]}")
            print(f"   [Debug] 缩放后坐标示例: {pixel_coords[0]}")
        # ----------------------------------

        valid_coords = []
        for x, y in top_left_coords:
            # 这里的判断非常严格，只要有一个角出界就丢弃
            if x >= 0 and y >= 0 and x + SRC_SIZE <= img_w and y + SRC_SIZE <= img_h:
                valid_coords.append([x, y])
        
        valid_coords = np.array(valid_coords)

        if len(valid_coords) == 0:
            print(f"   ❌ 错误：有效 Patch 数量为 0！请检查缩放因子是否正确。")
            return False

        # 保存
        os.makedirs(PATCHES_SAVE_DIR, exist_ok=True)
        h5_save_path = os.path.join(PATCHES_SAVE_DIR, f'{slide_id}.h5')
        
        with h5py.File(h5_save_path, 'w') as f:
            dset = f.create_dataset('coords', data=valid_coords)
            dset.attrs['patch_size'] = SRC_SIZE
            dset.attrs['patch_level'] = 0
            f.attrs['n_patches'] = len(valid_coords)
            
        print(f"   ✅ 保存成功: {len(valid_coords)} patches -> {os.path.basename(h5_save_path)}")
        return True

    except Exception as e:
        print(f"❌ 错误 {slide_id}: {e}")
        return False

# ================= 执行逻辑 =================
if __name__ == "__main__":
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        if 'slide_id' in df.columns:
            ids = df['slide_id'].tolist()
        else:
            ids = df.iloc[:, 0].tolist()
        
        print(f"🚀 开始处理 {len(ids)} 个样本...")
        
        for slide_id in tqdm(ids):
            # 这里的名字匹配非常重要，确保你的文件叫 SampleID.h5ad 和 SampleID.png
            h5ad_file = os.path.join(H5AD_DIR, f"{slide_id}.h5ad")
            img_file = os.path.join(IMAGE_DIR, f"{slide_id}.png") # 确保你的图片已经去掉后缀了
            
            process_single_sample(str(slide_id), h5ad_file, img_file)
    else:
        print("CSV not found.")

🚀 开始处理 65 个样本...


  2%|▏         | 1/65 [00:00<00:24,  2.57it/s]

   Using hires scale factor: 0.0728
   [Debug] 图片尺寸: 2000x1834
   [Debug] 原始坐标示例: [7453 3734]
   [Debug] 缩放后坐标示例: [542 271]
   ✅ 保存成功: 4992 patches -> ProstateAcinarCancer_10x.h5


  3%|▎         | 2/65 [00:00<00:21,  2.90it/s]

   Using hires scale factor: 0.0561
   [Debug] 图片尺寸: 2000x1650
   [Debug] 原始坐标示例: [26638 26818]
   [Debug] 缩放后坐标示例: [1494 1505]
   ✅ 保存成功: 4415 patches -> LungCancer_10x.h5


...

   Using hires scale factor: 0.0411
   [Debug] 图片尺寸: 2000x1937
   [Debug] 原始坐标示例: [10038  2638]
   [Debug] 缩放后坐标示例: [412 108]
   ✅ 保存成功: 407 patches -> GSM5621965.h5


 98%|█████████▊| 64/65 [00:21<00:00,  2.74it/s]

   Using hires scale factor: 0.1500
   [Debug] 图片尺寸: 2000x2000
   [Debug] 原始坐标示例: [4164 1313]
   [Debug] 缩放后坐标示例: [624 196]
   ✅ 保存成功: 4672 patches -> ColorectalCancer_10x.h5


100%|██████████| 65/65 [00:21<00:00,  2.98it/s]

   Using hires scale factor: 0.1500
   [Debug] 图片尺寸: 2000x2000
   [Debug] 原始坐标示例: [9791 8468]
   [Debug] 缩放后坐标示例: [1468 1270]
   ✅ 保存成功: 3639 patches -> 151673.h5
